# Week 1 Lab: Transformer Mechanics & Tokenization
**Research Engineer:** Victor Okoye
**Goal:** Probe a pretrained tokenizer (`o200k_base` / GPT-4o) to observe edge cases, the "typo tax", and the efficiency disparity between Western and non-Western languages.

In [1]:
# Install the tokenizer library
!pip install tiktoken

## 1. Environment Setup
First, we define a reusable function to encode a text string into integers and decode those integers back into byte-strings. This allows us to see exactly how the model slices up the input.

In [2]:
import tiktoken

def probe_tokenizer(text_string, model_name="gpt-4o"):
    encoding = tiktoken.encoding_for_model(model_name)
    token_integers = encoding.encode(text_string)
    token_bytes = [encoding.decode_single_token_bytes(token) for token in token_integers]
    
    print(f"\n--- Probing: '{text_string}' ---")
    print(f"Total Tokens: {len(token_integers)}")
    
    for i in range(len(token_integers)):
        chunk = token_bytes[i].decode("utf-8", errors="replace")
        print(f"Token {token_integers[i]}: '{chunk}'")

## 2. Baseline & Formatting Inefficiencies
In standard English, the tokenizer is highly efficient. However, unpredictable formatting (like trailing spaces) forces the model to generate entirely separate tokens just to represent empty space.

In [3]:
probe_tokenizer("Hello world")
probe_tokenizer("Hello world   ")


--- Probing: 'Hello world' ---
Total Tokens: 2
Token 13225: 'Hello'
Token 2375: ' world'

--- Probing: 'Hello world   ' ---
Total Tokens: 3
Token 13225: 'Hello'
Token 2375: ' world'
Token 271: '   '


## 4. The Typo Tax & Edge Cases
The tokenizer lacks contextual intelligence, so it performs blind dictionary lookups. If an input contains typos, missed apostrophes, or custom made-up words, the token count balloons rapidly, inflating processing costs before the Transformer even begins its attention calculations.

In [4]:
probe_tokenizer("i wan't you t elp me run tis code")
probe_tokenizer("codehashira")


--- Probing: 'i wan't you t elp me run tis code' ---
Total Tokens: 11
Token 72: 'i'
Token 20799: ' wan'
Token 1507: ''t'
Token 481: ' you'
Token 260: ' t'
Token 650: ' el'
Token 79: 'p'
Token 668: ' me'
Token 2461: ' run'
Token 65769: ' tis'
Token 3490: ' code'

--- Probing: 'codehashira' ---
Total Tokens: 3
Token 3056: 'code'
Token 16909: 'hash'
Token 2633: 'ira'


## 5. Architectural Takeaways for Applied AI
1. **Sanitize Inputs:** Running lightweight spell-checks before passing user prompts to an LLM reduces payload size and lowers API costs.
2. **Language Constraints:** Building automated systems (like WhatsApp bots) for local Nigerian demographics requires budgeting for higher token usage per interaction compared to English-only platforms.